# Original Diffusion-LM baseline on Shakespeare

This notebook implements the point-embedding version of continuous Diffusion-LM. Each character has one learned embedding `E[w]`; the clean continuous state is sampled as `x0 = E[w] + sigma0 * epsilon`. The model then learns to predict `x0` from a noisy diffusion state and to round its prediction back to a character.

It is the control experiment for `GaussianCodebookDiffusionLM` in `model.py`: this baseline has **no learned per-token variance** and **no aggregate-prior loss**.

In [ ]:
from pathlib import Path
from dataclasses import dataclass
import math

import torch
from torch import Tensor, nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

from model import ModelConfig, PointEmbeddingDiffusionLM

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(7)
print(device)

In [ ]:
# Character data and non-overlapping 64-character blocks.
data_dir = Path("shakespeare-dataset-main/text")
text = "\n".join(path.read_text() for path in sorted(data_dir.glob("*.txt")))
chars = sorted(set(text))
stoi = {char: index for index, char in enumerate(chars)}
itos = {index: char for char, index in stoi.items()}
tokens = torch.tensor([stoi[char] for char in text], dtype=torch.long)

class CharacterBlocks(Dataset):
    def __init__(self, values: Tensor, block_size: int, stride: int | None = None):
        self.values = values
        self.block_size = block_size
        self.stride = stride or block_size
        self.starts = range(0, len(values) - block_size + 1, self.stride)

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, index):
        start = self.starts[index]
        return self.values[start : start + self.block_size]

BLOCK_SIZE = 64
train_loader = DataLoader(CharacterBlocks(tokens, BLOCK_SIZE), batch_size=512, shuffle=True)

print(f"{len(text):,} characters | vocabulary: {len(chars)} | train blocks: {len(train_loader.dataset):,}")

5,330,377 characters | vocabulary: 84 | train blocks: 83,287


In [ ]:
class OriginalDiffusionLM(nn.Module):
    """Diffusion-LM with point embeddings and fixed x0 Gaussian noise."""

    def __init__(self, config: ModelConfig, x0_noise_std: float = 0.01):
        super().__init__()
        self.config = config
        self.x0_noise_std = x0_noise_std
        self.token_embedding = nn.Embedding(config.vocab_size, config.dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(config.dim, 4 * config.dim), nn.SiLU(), nn.Linear(4 * config.dim, config.dim)
        )
        self.blocks = nn.ModuleList(TransformerBlock(config) for _ in range(config.layers))
        self.final_norm = nn.LayerNorm(config.dim)
        self.x0_head = nn.Linear(config.dim, config.dim)
        self.rounding_head = nn.Linear(config.dim, config.vocab_size)
        self.register_buffer("alpha_bar", cosine_alpha_bar(config.diffusion_steps))

    def q_sample(self, x0: Tensor, timesteps: Tensor, noise: Tensor | None = None) -> Tensor:
        noise = torch.randn_like(x0) if noise is None else noise
        alpha = self.alpha_bar[timesteps].to(x0.dtype).view(-1, 1, 1)
        return alpha.sqrt() * x0 + (1 - alpha).sqrt() * noise

    def forward(self, x_t: Tensor, timesteps: Tensor) -> tuple[Tensor, Tensor]:
        time = self.time_mlp(sinusoidal_timestep_embedding(timesteps, self.config.dim).to(x_t.dtype)).unsqueeze(1)
        hidden = x_t + time
        for block in self.blocks:
            hidden = block(hidden)
        x0_hat = self.x0_head(self.final_norm(hidden))
        return x0_hat, self.rounding_head(x0_hat)

    def loss(self, token_ids: Tensor) -> tuple[Tensor, Tensor, Tensor]:
        x0 = self.token_embedding(token_ids)
        x0 = x0 + self.x0_noise_std * torch.randn_like(x0)
        timesteps = torch.randint(1, self.config.diffusion_steps + 1, (token_ids.size(0),), device=token_ids.device)
        x_t = self.q_sample(x0, timesteps)
        x0_hat, logits = self(x_t, timesteps)
        diffusion = F.mse_loss(x0_hat, x0)
        rounding = F.cross_entropy(logits.flatten(0, 1), token_ids.flatten())
        return diffusion + rounding, diffusion.detach(), rounding.detach()

    @torch.no_grad()
    def sample(self, batch_size: int = 1, reverse_steps: int = 50, clamp: bool = False) -> Tensor:
        self.eval()
        x_t = torch.randn(batch_size, self.config.sequence_length, self.config.dim, device=self.alpha_bar.device)
        times = torch.linspace(self.config.diffusion_steps, 0, reverse_steps + 1, device=x_t.device).long()
        for current, next_time in zip(times[:-1], times[1:]):
            t = torch.full((batch_size,), current, device=x_t.device, dtype=torch.long)
            x0_hat, logits = self(x_t, t)
            if clamp:
                ids = logits.argmax(dim=-1)
                x0_hat = self.token_embedding(ids)
            alpha_t = self.alpha_bar[current].to(x_t.dtype)
            alpha_next = self.alpha_bar[next_time].to(x_t.dtype)
            eps_hat = (x_t - alpha_t.sqrt() * x0_hat) / (1 - alpha_t).sqrt().clamp_min(1e-8)
            x_t = alpha_next.sqrt() * x0_hat + (1 - alpha_next).sqrt() * eps_hat
        _, logits = self(x_t, torch.zeros(batch_size, device=x_t.device, dtype=torch.long))
        return logits.argmax(dim=-1)

In [ ]:
config = ModelConfig(vocab_size=len(chars), sequence_length=BLOCK_SIZE)
# This is the shared implementation in model.py.  Its denoiser is exactly
# the one used by GaussianCodebookDiffusionLM; only the codebook differs.
model = PointEmbeddingDiffusionLM(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
print(f"parameters: {sum(parameter.numel() for parameter in model.parameters()):,}")

parameters: 5,379,620


In [ ]:
def run_epoch(loader, training: bool, log_every: int = 50):
    model.train(training)
    totals = torch.zeros(3)
    for step, batch in enumerate(loader, start=1):
        batch = batch.to(device)
        with torch.set_grad_enabled(training):
            loss_terms = model.training_loss(batch)
            loss, diffusion, rounding = loss_terms.total, loss_terms.diffusion, loss_terms.rounding
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        totals += torch.tensor([loss.item(), diffusion.item(), rounding.item()])
        if training and (step % log_every == 0 or step == len(loader)):
            running = totals / step
            print(f"  step {step:>4}/{len(loader)} | loss={loss.item():.4f} | running loss={running[0]:.4f} mse={running[1]:.4f} ce={running[2]:.4f}")
    return totals / len(loader)

# Start with one epoch as a smoke test; then increase this to 20-50.
for epoch in range(1, 10):
    train_metrics = run_epoch(train_loader, training=True)
    print(f"epoch {epoch:02d} | train loss={train_metrics[0]:.4f} mse={train_metrics[1]:.4f} ce={train_metrics[2]:.4f}")

  step   50/163 | loss=1.2633 | running loss=2.2697 mse=1.0287 ce=1.2409
  step  100/163 | loss=0.8221 | running loss=1.6394 mse=0.7764 ce=0.8629
  step  150/163 | loss=0.6426 | running loss=1.3424 mse=0.6315 ce=0.7109
  step  163/163 | loss=0.7710 | running loss=1.2879 mse=0.6029 ce=0.6850
epoch 01 | train loss=1.2879 mse=0.6029 ce=0.6850
  step   50/163 | loss=0.5560 | running loss=0.6104 mse=0.2395 ce=0.3709
  step  100/163 | loss=0.4980 | running loss=0.5835 mse=0.2203 ce=0.3632
  step  150/163 | loss=0.5306 | running loss=0.5588 mse=0.2033 ce=0.3555
  step  163/163 | loss=0.4639 | running loss=0.5558 mse=0.2003 ce=0.3555
epoch 02 | train loss=0.5558 mse=0.2003 ce=0.3555
  step   50/163 | loss=0.4247 | running loss=0.4777 mse=0.1535 ce=0.3242
  step  100/163 | loss=0.4674 | running loss=0.4755 mse=0.1495 ce=0.3260
  step  150/163 | loss=0.4756 | running loss=0.4751 mse=0.1484 ce=0.3267
  step  163/163 | loss=0.4102 | running loss=0.4754 mse=0.1485 ce=0.3269
epoch 03 | train loss=0.

In [ ]:
# DDIM-style deterministic reverse sampling. `clamp=True` is the optional Diffusion-LM clamping ablation.
sample_ids = model.sample(batch_size=2, reverse_steps=1000, clamp=False).cpu()
for sample in sample_ids:
    print("".join(itos[token.item()] for token in sample))
    print()

NameError: name 'model' is not defined